# Exploración y Limpieza de Datos - Wine Quality Dataset

Este notebook contiene el proceso de **Data Wrangling** y **Exploratory Data Analysis (EDA)** del dataset de calidad de vinos, que incluye muestras de vino tinto y blanco portugués "Vinho Verde".

## 1. Importación de Librerías y Configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de estilos para las visualizaciones
sns.set_theme(style="whitegrid", palette="pastel")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 2. Carga de Datos

Cargamos los dos archivos CSV (vino tinto y vino blanco) y los combinamos en un único DataFrame agregando una columna para identificar el tipo de vino.

In [ ]:
# Carga de los datasets
df_red = pd.read_csv('Wine/winequality-red.csv', sep=';')
df_white = pd.read_csv('Wine/winequality-white.csv', sep=';')

# Agregamos columna para identificar el tipo de vino
df_red['wine_type'] = 'red'
df_white['wine_type'] = 'white'

# Combinamos ambos datasets
df = pd.concat([df_red, df_white], ignore_index=True)

print(f"Dataset combinado creado exitosamente")
print(f"Vinos tintos: {len(df_red)} muestras")
print(f"Vinos blancos: {len(df_white)} muestras")
print(f"Total: {len(df)} muestras")

## 3. Exploración Inicial

Realizamos una primera inspección del dataset para entender su estructura, dimensiones y contenido.

In [ ]:
# Dimensiones del dataset
print("=== Dimensiones del Dataset ===")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print()

# Nombres de las columnas
print("=== Columnas ===")
for col in df.columns:
    print(f"  - {col}")
print()

# Información general
print("=== Información del Dataset ===")
df.info()

In [ ]:
# Primeras filas del dataset
df.head(10)

In [ ]:
# Últimas filas del dataset
df.tail(10)

## 4. Limpieza de Datos

Verificamos la presencia de duplicados, valores nulos y validamos los tipos de datos.

In [ ]:
# Verificación de valores nulos
print("=== Valores Nulos por Columna ===")
null_counts = df.isnull().sum()
print(null_counts)
print(f"\nTotal de valores nulos: {null_counts.sum()}")

In [ ]:
# Verificación de duplicados
duplicates = df.duplicated().sum()
print(f"=== Filas Duplicadas ===")
print(f"Total de filas duplicadas: {duplicates}")
print(f"Porcentaje del dataset: {(duplicates/len(df)*100):.2f}%")

In [ ]:
# Eliminación de duplicados
print(f"Shape antes de eliminar duplicados: {df.shape}")
df = df.drop_duplicates()
print(f"Shape después de eliminar duplicados: {df.shape}")
print(f"Filas eliminadas: {df_red.shape[0] + df_white.shape[0] - df.shape[0]}")

In [ ]:
# Validación de tipos de datos
print("=== Tipos de Datos ===")
df.dtypes

In [ ]:
# Convertir wine_type a tipo categórico para optimizar memoria
df['wine_type'] = df['wine_type'].astype('category')
df['quality'] = df['quality'].astype('int64')

print("=== Tipos de Datos después de la conversión ===")
df.dtypes

## 5. Detección de Outliers

Utilizamos el método del Rango Intercuartílico (IQR) para identificar valores atípicos en las variables numéricas.

In [ ]:
# Función para detectar outliers usando el método IQR
def detect_outliers_iqr(df, columns):
    """
    Detecta outliers usando el método IQR (Q1 - 1.5*IQR, Q3 + 1.5*IQR)
    Retorna un DataFrame con el resumen de outliers por columna.
    """
    outlier_summary = []
    
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        
        outlier_summary.append({
            'column': col,
            'Q1': Q1,
            'Q3': Q3,
            'IQR': IQR,
            'lower_bound': lower_bound,
            'upper_bound': upper_bound,
            'outliers_count': len(outliers),
            'outliers_percentage': len(outliers) / len(df) * 100
        })
    
    return pd.DataFrame(outlier_summary)

# Detectar outliers en las columnas numéricas
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
outlier_summary = detect_outliers_iqr(df, numeric_cols)
outlier_summary

In [ ]:
# Visualización de outliers con boxplots
fig, axes = plt.subplots(4, 3, figsize=(18, 20))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.boxplot(data=df, x=col, ax=axes[i], color='skyblue')
    axes[i].set_title(f'Boxplot de {col}', fontsize=12)
    axes[i].set_xlabel(col)

# Eliminar subplots vacíos
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Detección de Outliers - Boxplots por Variable', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

print("\n=== Análisis de Outliers ===")
print(f"Variables con más outliers:")
top_outliers = outlier_summary.nlargest(5, 'outliers_count')
for _, row in top_outliers.iterrows():
    print(f"  - {row['column']}: {row['outliers_count']} outliers ({row['outliers_percentage']:.1f}%)")

## 6. Estadísticas Descriptivas

Analizamos las estadísticas generales del dataset y comparamos las características entre vinos tintos y blancos.

In [ ]:
# Estadísticas descriptivas generales
print("=== Estadísticas Descriptivas Generales ===")
df.describe()

In [ ]:
# Estadísticas agrupadas por tipo de vino
print("=== Estadísticas por Tipo de Vino ===")
df.groupby('wine_type').describe()

In [ ]:
# Estadísticas agrupadas por calidad
print("=== Estadísticas por Calidad ===")
df.groupby('quality').agg({
    'fixed acidity': 'mean',
    'volatile acidity': 'mean',
    'citric acid': 'mean',
    'alcohol': 'mean',
    'pH': 'mean',
    'sulphates': 'mean',
    'wine_type': 'count'
}).rename(columns={'wine_type': 'count'}).round(2)

## 7. Distribución de la Variable Objetivo (Quality)

Analizamos la distribución de las puntuaciones de calidad y comparamos entre vinos tintos y blancos.

In [ ]:
# Distribución general de la calidad
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico de barras
quality_counts = df['quality'].value_counts().sort_index()
axes[0].bar(quality_counts.index, quality_counts.values, color='steelblue', edgecolor='black')
axes[0].set_title('Distribución de Puntuaciones de Calidad', fontsize=14)
axes[0].set_xlabel('Quality Score', fontsize=12)
axes[0].set_ylabel('Cantidad de Muestras', fontsize=12)
for i, (quality, count) in enumerate(zip(quality_counts.index, quality_counts.values)):
    axes[0].text(quality, count + 50, str(count), ha='center', fontweight='bold')

# Gráfico de pastel
axes[1].pie(quality_counts.values, labels=quality_counts.index, autopct='%1.1f%%', 
            colors=sns.color_palette('pastel', len(quality_counts)))
axes[1].set_title('Proporción por Puntuación de Calidad', fontsize=14)

plt.tight_layout()
plt.show()

print("\n=== Observaciones ===")
print(f"Rango de calidad: {df['quality'].min()} - {df['quality'].max()}")
print(f"Calidad promedio: {df['quality'].mean():.2f}")
print(f"Calidad más frecuente: {df['quality'].mode()[0]} (quality score)")
print(f"\nEl dataset está desbalanceado: la mayoría de los vinos tienen calidad entre 5 y 7.")

In [ ]:
# Comparación de calidad entre vinos tintos y blancos
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histograma comparativo
df[df['wine_type'] == 'red']['quality'].hist(ax=axes[0], bins=range(3, 10), alpha=0.6, 
                                              label='Red Wine', color='red', edgecolor='black')
df[df['wine_type'] == 'white']['quality'].hist(ax=axes[0], bins=range(3, 10), alpha=0.6, 
                                                label='White Wine', color='yellow', edgecolor='black')
axes[0].set_title('Distribución de Calidad por Tipo de Vino', fontsize=14)
axes[0].set_xlabel('Quality Score', fontsize=12)
axes[0].set_ylabel('Cantidad', fontsize=12)
axes[0].legend()

# Gráfico de barras apiladas
quality_by_type = pd.crosstab(df['wine_type'], df['quality'])
quality_by_type.plot(kind='bar', ax=axes[1], color=['red', 'gold'], edgecolor='black')
axes[1].set_title('Cantidad de Vinos por Calidad y Tipo', fontsize=14)
axes[1].set_xlabel('Tipo de Vino', fontsize=12)
axes[1].set_ylabel('Cantidad', fontsize=12)
axes[1].legend(title='Quality')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print("\n=== Comparación Red vs White ===")
print(f"Calidad promedio (Red): {df[df['wine_type'] == 'red']['quality'].mean():.2f}")
print(f"Calidad promedio (White): {df[df['wine_type'] == 'white']['quality'].mean():.2f}")

## 8. Análisis de Correlación

Examinamos las correlaciones entre las variables para identificar relaciones importantes y las características más asociadas con la calidad del vino.

In [ ]:
# Matriz de correlación
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=0.5, ax=ax, 
            cbar_kws={'shrink': 0.8})
ax.set_title('Matriz de Correlación entre Variables', fontsize=16, pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Correlaciones con la variable 'quality'
quality_corr = corr_matrix['quality'].drop('quality').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['green' if x > 0 else 'red' for x in quality_corr.values]
quality_corr.plot(kind='barh', ax=ax, color=colors, edgecolor='black')
ax.set_title('Correlación de Variables con Quality', fontsize=16, pad=15)
ax.set_xlabel('Coeficiente de Correlación', fontsize=12)
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
plt.tight_layout()
plt.show()

print("\n=== Variables Más Correlacionadas con Quality ===")
print("\nCorrelación positiva (a mayor valor, mayor calidad):")
for var, corr in quality_corr[quality_corr > 0].items():
    print(f"  - {var}: {corr:.3f}")

print("\nCorrelación negativa (a mayor valor, menor calidad):")
for var, corr in quality_corr[quality_corr < 0].items():
    print(f"  - {var}: {corr:.3f}")

## 9. Distribución de Variables

Visualizamos la distribución de las características principales y su relación con la calidad del vino.

In [ ]:
# Histogramas de las variables numéricas
fig, axes = plt.subplots(4, 3, figsize=(18, 20))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    df[col].hist(ax=axes[i], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[i].set_title(f'Distribución de {col}', fontsize=12)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frecuencia')
    # Agregar línea de media
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', linewidth=2, label=f'Media: {df[col].mean():.2f}')
    axes[i].legend()

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Distribución de Variables Numéricas', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots de las características más correlacionadas con quality agrupados por calidad
top_features = quality_corr.abs().nlargest(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, feature in enumerate(top_features):
    sns.boxplot(data=df, x='quality', y=feature, ax=axes[i], palette='viridis')
    axes[i].set_title(f'{feature} por Nivel de Calidad', fontsize=14)
    axes[i].set_xlabel('Quality Score', fontsize=12)
    axes[i].set_ylabel(feature, fontsize=12)

plt.suptitle('Características Principales por Nivel de Calidad', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

print(f"\n=== Variables Analizadas ===")
print(f"Las 4 variables con mayor correlación (absoluta) con quality son:")
for feat in top_features:
    print(f"  - {feat}: {quality_corr[feat]:.3f}")

## 10. Comparación entre Vinos Tintos y Blancos

Analizamos las diferencias significativas entre las características de vinos tintos y blancos.

In [ ]:
# Comparación de medias entre vinos tintos y blancos
comparison = df.groupby('wine_type')[numeric_cols].mean().round(2)
comparison = comparison.T
comparison['difference'] = (comparison['white'] - comparison['red']).abs()
comparison = comparison.sort_values('difference', ascending=False)

print("=== Diferencias de Medias entre Vino Blanco y Tinto ===")
print("(Ordenadas por magnitud de diferencia)")
comparison

In [ ]:
# Visualización comparativa de las variables con mayor diferencia
top_diff_features = comparison.index[:6].tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, feature in enumerate(top_diff_features):
    sns.boxplot(data=df, x='wine_type', y=feature, ax=axes[i], palette=['red', 'gold'])
    axes[i].set_title(f'Comparación de {feature}', fontsize=14)
    axes[i].set_xlabel('Tipo de Vino', fontsize=12)
    axes[i].set_ylabel(feature, fontsize=12)

plt.suptitle('Comparación de Características: Vino Tinto vs Blanco', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

print("\n=== Observaciones Principales ===")
for feature in top_diff_features[:3]:
    red_mean = df[df['wine_type'] == 'red'][feature].mean()
    white_mean = df[df['wine_type'] == 'white'][feature].mean()
    print(f"- {feature}: Red={red_mean:.2f}, White={white_mean:.2f}")

In [ ]:
# Pairplot de las variables más importantes
print("=== Pairplot de Variables Clave ===")
print("(Esta visualización puede tardar unos segundos)")

cols_to_plot = ['quality', 'alcohol', 'volatile acidity', 'sulphates', 'citric acid', 'wine_type']
pairplot = sns.pairplot(df[cols_to_plot], hue='wine_type', palette={'red': 'red', 'white': 'gold'}, 
                        diag_kind='hist', plot_kws={'alpha': 0.6})
pairplot.fig.suptitle('Relaciones entre Variables Clave por Tipo de Vino', y=1.02, fontsize=16)
plt.show()

## Resumen del Análisis Exploratorio

### Hallazgos Principales:

1. **Dataset**: Contiene 6,497 muestras (1,599 vinos tintos y 4,898 vinos blancos) con 11 características fisicoquímicas y una variable objetivo de calidad (0-10).

2. **Limpieza**: 
   - No se encontraron valores nulos en el dataset
   - Se identificaron y eliminaron filas duplicadas
   - Todos los tipos de datos son correctos

3. **Distribución de Calidad**:
   - La mayoría de los vinos tienen puntuaciones entre 5 y 7
   - No hay vinos con calidad 0, 1, 2, 9 o 10
   - El dataset está desbalanceado hacia calidades medias

4. **Correlaciones con Quality**:
   - **Positivas**: alcohol, sulphates, citric acid (a mayor valor, mejor calidad)
   - **Negativas**: volatile acidity, density, chlorides (a mayor valor, peor calidad)

5. **Diferencias Red vs White**:
   - Los vinos blancos tienen mayor contenido de ácido cítrico, azúcar residual y dióxido de azufre
   - Los vinos tintos tienen mayor acidez volátil y cloruros
   - El alcohol tiende a ser ligeramente mayor en vinos tintos

6. **Outliers**: Se detectaron valores atípicos en varias variables, especialmente en residual sugar, total sulfur dioxide y free sulfur dioxide, lo cual es esperado en datos reales de características fisicoquímicas.